In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

# --------------------------------------------------
# 1. Create SparkSession
# --------------------------------------------------
# SparkSession is the main entry point for working with
# Spark DataFrames and Spark SQL.
#
# enableHiveSupport() enables integration with the Hive
# metastore/catalog for managing tables and metadata.
spark = SparkSession.builder \
    .appName("Spark with Hive") \
    .enableHiveSupport() \
    .getOrCreate()


# --------------------------------------------------
# 2. Create sample data
# --------------------------------------------------
# This is a normal Python list containing row data.
# The data is converted into a Spark DataFrame below.
#
# datetime.strptime() converts a string into a Python
# datetime object using the specified date/time format.
data = [
    ["Product A", 1001, datetime.strptime("2023-07-20", "%Y-%m-%d"), datetime.strptime("2023-07-20 10:15:30", "%Y-%m-%d %H:%M:%S"), 29.99],
    ["Product B", 1002, datetime.strptime("2023-07-19", "%Y-%m-%d"), datetime.strptime("2023-07-19 14:20:45", "%Y-%m-%d %H:%M:%S"), 49.99],
    ["Product C", 1003, datetime.strptime("2023-07-18", "%Y-%m-%d"), datetime.strptime("2023-07-18 09:30:15", "%Y-%m-%d %H:%M:%S"), 39.99],
    ["Product D", 1004, datetime.strptime("2023-07-17", "%Y-%m-%d"), datetime.strptime("2023-07-17 16:45:00", "%Y-%m-%d %H:%M:%S"), 19.99]
]


# --------------------------------------------------
# 3. Define DataFrame schema explicitly
# --------------------------------------------------
# StructType defines the complete DataFrame schema.
# Each StructField defines: column name, data type,
# and whether NULL values are allowed.
#
# Explicit schemas are useful when we already know the
# expected structure and want control over data types.
schema = StructType([
    StructField("Product", StringType(), True),
    StructField("ID", IntegerType(), True),
    StructField("Date", DateType(), True),
    StructField("Timestamp", TimestampType(), True),
    StructField("Price", FloatType(), True)
])


# --------------------------------------------------
# 4. Create Spark DataFrame
# --------------------------------------------------
# Converts the Python data into a Spark DataFrame
# using the explicitly defined schema.
df = spark.createDataFrame(data, schema)


# Display the DataFrame schema
df.printSchema()

# Display the DataFrame records
# show() is an action that triggers Spark execution.
df.show()

root
 |-- Product: string (nullable = true)
 |-- ID: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Price: float (nullable = true)

+---------+----+----------+-------------------+-----+
|  Product|  ID|      Date|          Timestamp|Price|
+---------+----+----------+-------------------+-----+
|Product A|1001|2023-07-20|2023-07-20 10:15:30|29.99|
|Product B|1002|2023-07-19|2023-07-19 14:20:45|49.99|
|Product C|1003|2023-07-18|2023-07-18 09:30:15|39.99|
|Product D|1004|2023-07-17|2023-07-17 16:45:00|19.99|
+---------+----+----------+-------------------+-----+



## Reading Data

In [ ]:
# First read example:
# Read CSV data without schema inference and provide an explicit schema.
# The header row is used as column names and is not treated as data.

# Define the schema explicitly so Spark does not need to infer
# column names and data types by scanning the CSV file.
schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("OrderItemID", IntegerType(), True),
    StructField("ProductID", StringType(), True),
    StructField("SellerID", StringType(), True),
    StructField("ShippingLimitDate", TimestampType(), True),
    StructField("Price", DoubleType(), True),
    StructField("FreightValue", DoubleType(), True)
])

# Path of the CSV file
hdfs_path = '/tmp/spark_datasets/order_items_dataset.csv'

# Read the CSV file using the explicitly defined schema.
#
# header=True:
#   Treats the first row of the CSV as column names.
#
# inferSchema=False:
#   Do not automatically determine the data types.
#   The data types will come from the schema defined above.
#
# schema(schema):
#   Applies our explicit StructType schema to the DataFrame.
df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'false') \
    .schema(schema) \
    .load(hdfs_path)

# Display the DataFrame schema to verify the column names and data types.
df.printSchema()

# Display the first 5 records.
df.show(5)

root
 |-- OrderID: string (nullable = true)
 |-- OrderItemID: integer (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- SellerID: string (nullable = true)
 |-- ShippingLimitDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- FreightValue: double (nullable = true)



+--------------------+-----------+--------------------+--------------------+-------------------+-----+------------+
|             OrderID|OrderItemID|           ProductID|            SellerID|  ShippingLimitDate|Price|FreightValue|
+--------------------+-----------+--------------------+--------------------+-------------------+-----+------------+
|00010242fe8c5a6d1...|          1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|       13.29|
|00018f77f2f0320c5...|          1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|       19.93|
|000229ec398224ef6...|          1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|       17.87|
|00024acbcdf0a6daa...|          1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|       12.79|
|00042b26cf59d7ce6...|          1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|       18.14|
+--------------------+-----------+--------------------+-----------------

In [ ]:
# Second read example:
# Infer the schema automatically instead of defining StructType manually.
# header=True → first row is treated as column names.
# inferSchema=True → Spark scans the data to determine appropriate data types.

hdfs_path = '/tmp/spark_datasets/order_items_dataset.csv'

# File path pattern when reading from a different Hadoop cluster
# hdfs_path = 'hdfs://<namenode_host_ip_address>:<port>/tmp/spark_datasets/order_items_dataset.csv'

df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load(hdfs_path)

# Verify the schema inferred by Spark
df.printSchema()

# Display the DataFrame
df.show()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 

## Checking Partitions and Partition Size

In [ ]:
# Check Spark's default parallelism.
# This value represents the default level of parallelism used
# for certain Spark operations when the number of partitions
# is not explicitly specified.

print(spark.sparkContext.defaultParallelism)

2


In [ ]:
# Check the maximum number of bytes that Spark will use
# when creating a partition while reading file-based data.
# File -> Spark reads file -> Splits input into partitions -> Each partition can be up to the configured maximum input size
# The important word is maximum. It doesn't mean every partition will actually contain exactly 128 MiB.

print(spark.conf.get("spark.sql.files.maxPartitionBytes"))

134217728b


In [ ]:
# Check the current number of partitions in the DataFrame.
print(f'Number of partitions: {df.rdd.getNumPartitions()}')

# repartition() returns a new DataFrame with the specified
# number of partitions and involves a shuffle to redistribute data.
# repartition is a transformation
df_new = df.repartition(5)

# Verify the new partition count.
print(f'Number of partitions: {df_new.rdd.getNumPartitions()}')

Number of partitions: 2


Number of partitions: 5


## Doing Basic SELECT

In [ ]:
# Select columns in different options
from pyspark.sql.functions import *

# select() creates a new DataFrame containing only the specified columns.
df.select('order_id').show(5) 

# this creates a new dataframe with only the order_id and shipping_limit_date columns
# You can simply pass multiple column names, The order you specify them in is the order they'll appear in the resulting DataFrame.
df.select('order_id', 'shipping_limit_date').show(5) 

# col() creates a Spark Column expression referring to a DataFrame column.
# The col() form becomes especially useful when you start doing things like:
'''
col("price") > 100
col("price") * 1.18
col("name").alias("customer_name")
'''
df.select(
    col('order_id'),
    col('shipping_limit_date')
).show(5) 

# alias() gives the selected column a new name in the resulting DataFrame.
# It does not modify the original DataFrame.
df.select(
    col('order_id').alias('oid'),
    col('shipping_limit_date').alias('limit_date')
).show(5)

+--------------------+
|            order_id|
+--------------------+
|00010242fe8c5a6d1...|
|00018f77f2f0320c5...|
|000229ec398224ef6...|
|00024acbcdf0a6daa...|
|00042b26cf59d7ce6...|
+--------------------+
only showing top 5 rows

+--------------------+-------------------+
|            order_id|shipping_limit_date|
+--------------------+-------------------+
|00010242fe8c5a6d1...|2017-09-19 09:45:35|
|00018f77f2f0320c5...|2017-05-03 11:05:13|
|000229ec398224ef6...|2018-01-18 14:48:30|
|00024acbcdf0a6daa...|2018-08-15 10:10:18|
|00042b26cf59d7ce6...|2017-02-13 13:57:51|
+--------------------+-------------------+
only showing top 5 rows

+--------------------+-------------------+
|            order_id|shipping_limit_date|
+--------------------+-------------------+
|00010242fe8c5a6d1...|2017-09-19 09:45:35|
|00018f77f2f0320c5...|2017-05-03 11:05:13|
|000229ec398224ef6...|2018-01-18 14:48:30|
|00024acbcdf0a6daa...|2018-08-15 10:10:18|
|00042b26cf59d7ce6...|2017-02-13 13:57:51|
+-----------

## Deriving New Columns

In [ ]:
# Derive new columns using withColumn()
# Syntax: df.withColumn("new_column", expression)
# Create a new DataFrame with a column derived from an expression
# withColumn() returns a new DataFrame; it does not modify the original DataFrame.

# Extract year and month from the shipping_limit_date column.
# col() creates a Spark Column expression used by the year() and month() functions.
# In below example we are chaining withColumn function to create two different columns.

df2 = df.withColumn(
    "year",
    year(col("shipping_limit_date"))
).withColumn(
    "month",
    month(col("shipping_limit_date"))
)

# So if do df.show(), then we won't see the new columns year and month because dataframes are immutable 
# That is why need to assign the result of withColumn to a new dataframe
# So when we do df2.show(), then we will see the new columns year and month.

df2.select("order_id", "shipping_limit_date", "year", "month").show(5)

+--------------------+-------------------+----+-----+
|            order_id|shipping_limit_date|year|month|
+--------------------+-------------------+----+-----+
|00010242fe8c5a6d1...|2017-09-19 09:45:35|2017|    9|
|00018f77f2f0320c5...|2017-05-03 11:05:13|2017|    5|
|000229ec398224ef6...|2018-01-18 14:48:30|2018|    1|
|00024acbcdf0a6daa...|2018-08-15 10:10:18|2018|    8|
|00042b26cf59d7ce6...|2017-02-13 13:57:51|2017|    2|
+--------------------+-------------------+----+-----+
only showing top 5 rows



In [ ]:
# Rename an existing column in the DataFrame.
# withColumnRenamed() changes only the column name;
# the underlying data and datatype remain unchanged.

df3 = df2.withColumnRenamed(
    "shipping_limit_date",
    "shipping_limit_datetime"
)

# Display the order ID and the renamed column.
df3.select(
    "order_id",
    "shipping_limit_datetime"
).show(5)

df3 = df2.withColumnRenamed('shipping_limit_date', 'shipping_limit_datetime')

df3.select("order_id", "shipping_limit_datetime").show(5)

+--------------------+-----------------------+
|            order_id|shipping_limit_datetime|
+--------------------+-----------------------+
|00010242fe8c5a6d1...|    2017-09-19 09:45:35|
|00018f77f2f0320c5...|    2017-05-03 11:05:13|
|000229ec398224ef6...|    2018-01-18 14:48:30|
|00024acbcdf0a6daa...|    2018-08-15 10:10:18|
|00042b26cf59d7ce6...|    2017-02-13 13:57:51|
+--------------------+-----------------------+
only showing top 5 rows



In [ ]:
# Filter rows based on a single condition.
# filter() returns a new DataFrame containing only rows
# that satisfy the specified condition.
df3.filter(
    col('order_id') == '00010242fe8c5a6d1ba2dd792cb16214'
).show(5)


# Filter rows where order_id matches any value in the given list.
# isin() is similar to the SQL IN operator.
order_ids = [
    '00010242fe8c5a6d1ba2dd792cb16214',
    '00018f77f2f0320c557190d7a144bdd3'
]

df3.filter(
    col('order_id').isin(order_ids)
).show(5)


# Filter using multiple conditions.
# & represents AND when combining Spark Column expressions.
df3.filter(
    (col('price') < 50) &
    (col('freight_value') < 10)
).show(5)


# SQL-style filter expression.
# The condition is written as a SQL expression string.
df3.filter(
    "price < 50 and freight_value < 10"
).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|    9|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|
+--------------------+-------------+--------------------+----

In [ ]:
# Drop the month column from the resulting DataFrame.
# drop() does not modify the original DataFrame because
# Spark DataFrames are immutable.
df3.drop('month').show(5)


+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13|239.9|        19.93|2017|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30|199.0|        17.87|2018|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|    2018-08-15 10:10:18|12.99|        12.79|2018|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|    2017-02-13 13:57:51|19

In [ ]:
#drop duplicates row based on multiple columns - duplicates are dropped based on the first occurrence of the row, 
# so if there are multiple rows with the same values in the specified columns, only the first one will be kept and the rest will be dropped.
# Remove duplicate rows based on the specified columns.
# Here, order_id + order_item_id together are used to
# determine whether rows are duplicates.
df3.dropDuplicates(
    ['order_id', 'order_item_id']
).show(5)

# order_id , order_item_id, c1, c2
#   1      ,    2        , A , B
#   1      ,    2        , A , B
#   1      ,    2        , C , D
#   1      ,    3        , E , F

# order_id , order_item_id, c1, c2
#   1      ,    2        , A , B
#   1      ,    3        , E , F

+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|year|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13| 239.9|        19.93|2017|    5|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30| 199.0|        17.87|2018|    1|
|00048cc3ae777c65d...|            1|ef92defde845ab845...|6426d21aca402a131...|    2017-05-23 03:55:27|  21.9|        12.69|2017|    5|
|0005a1a1728c9d785...|            1|310ae3c140ff94b03...|a416b6a846a117243...|    2018-03-26 18:31:29|145.95|        11.65|2018|    3|
|0005f50442cb953dc...|            1|4535b0e1091c278df..

In [ ]:
# Get distinct rows.
# When no columns are specified, the entire row is considered
# when identifying duplicates.
df3.dropDuplicates().show(5)


In [ ]:
# Arrange/sort the DataFrame by price in descending order.
# desc() → highest value first.
df3.orderBy(
    col('price').desc()
).show(5)


# Sort using multiple columns.
# First sort by price in ascending order.
# If multiple rows have the same price, sort those rows
# by freight_value in descending order.
df3.orderBy(
    col('price').asc(),
    col('freight_value').desc()
).show(5)


+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime| price|freight_value|year|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+------+-------------+----+-----+
|0812eb902a67711a1...|            1|489ae2aa008f02150...|e3b4998c7a498169d...|    2017-02-16 20:37:36|6735.0|       194.31|2017|    2|
|fefacc66af859508b...|            1|69c590f7ffc7bf8db...|80ceebb4ee9b31afb...|    2018-08-02 04:05:13|6729.0|       193.21|2018|    8|
|f5136e38d1a14a4db...|            1|1bdf5e6731585cf01...|ee27a8f15b1dded4d...|    2017-06-15 02:45:17|6499.0|       227.66|2017|    6|
|a96610ab360d42a2e...|            1|a6492cc69376c469a...|59417c56835dd8e2e...|    2017-04-18 13:25:18|4799.0|       151.34|2017|    4|
|199af31afc78c699f...|            1|c3ed642d592594bb6..

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+
|c5bdd8ef3c0ec4202...|            2|8a3254bee785a526d...|96804ea39d96eb908...|    2018-05-07 02:55:22| 0.85|         22.3|2018|    5|
|6e864b3f0ec710311...|            1|8a3254bee785a526d...|96804ea39d96eb908...|    2018-05-02 20:30:34| 0.85|        18.23|2018|    5|
|3ee6513ae7ea23bdf...|            1|8a3254bee785a526d...|96804ea39d96eb908...|    2018-05-04 03:55:26| 0.85|        18.23|2018|    5|
|8272b63d03f5f79c5...|            4|05b515fdc76e888aa...|2709af9587499e95e...|    2017-07-21 18:25:23|  1.2|         7.89|2017|    7|
|8272b63d03f5f79c5...|            2|05b515fdc76e888aa...|2709a

In [ ]:
# Group By Operation

# Group the data by year and calculate aggregate statistics
# for each year.
df3.groupBy('year').agg(
    count('*').alias('total_count'),
    avg('price').alias('avg_price'),
    sum('price').alias('sum_price'),
    min('price').alias('min_price'),
    max('price').alias('max_price')
).show(5)


# Group by multiple columns.
# Here, each unique year + month combination forms a group.
# The results are then sorted by year and month in ascending order.
df3.groupBy('year', 'month').agg(
    count('*').alias('total_count'),
    avg('price').alias('avg_price'),
    sum('price').alias('sum_price'),
    min('price').alias('min_price'),
    max('price').alias('max_price')
).orderBy(
    col('year').asc(),
    col('month').asc()
).show(20)

# Note: groupBy() usually requires a shuffle because records with the same grouping key need to be brought together for aggregation.

+----+-----------+------------------+-----------------+---------+---------+
|year|total_count|         avg_price|        sum_price|min_price|max_price|
+----+-----------+------------------+-----------------+---------+---------+
|2018|      62511|120.08515685240229| 7506643.24000052|     0.85|   6729.0|
|2017|      49765|121.26732804178806|6034868.579999583|      1.2|   6735.0|
|2020|          4|             86.49|           345.96|    69.99|    99.99|
|2016|        370|134.55654054054082| 49785.9200000001|      6.0|   1399.0|
+----+-----------+------------------+-----------------+---------+---------+



+----+-----+-----------+------------------+------------------+---------+---------+
|year|month|total_count|         avg_price|         sum_price|min_price|max_price|
+----+-----+-----------+------------------+------------------+---------+---------+
|2016|    9|          4|           48.6175|            194.47|    44.99|     59.5|
|2016|   10|        365|135.83712328767152|49580.550000000105|      6.0|   1399.0|
|2016|   12|          1|              10.9|              10.9|     10.9|     10.9|
|2017|    1|        681|117.65747430249674| 80124.74000000028|      2.9|   1999.0|
|2017|    2|       1866|131.82315648445825|245982.00999999908|      3.9|   6735.0|
|2017|    3|       2751| 124.7701199563779| 343242.5999999956|      4.9|   3999.9|
|2017|    4|       2364| 130.3503891708958| 308148.3199999977|      4.9|   4799.0|
|2017|    5|       4150| 121.8446891566257| 505655.4599999967|      3.5|   4690.0|
|2017|    6|       3801|123.38885819521052|469001.04999999516|     3.49|   6499.0|
|201

In [ ]:
# Create an accumulator with an initial value of 0.
# Accumulators allow tasks running on executors to add/update
# a shared aggregate value that can be read by the Driver.
accum = spark.sparkContext.accumulator(0)


# Apply a function to every row in the DataFrame.
# For each row, retrieve the price and add it to the accumulator.
df3.foreach(
    lambda row: accum.add(row['price'])
)


# Read the final accumulated value on the Driver.
print(accum.value)  # Accessed by the Driver

13591643.70000748


In [ ]:
# Create a new column called price_category based on the value of price.
# when() is used to define conditional logic, similar to SQL CASE WHEN.

# Syntax
'''
when(condition1, value1) \
    .when(condition2, value2) \
    .otherwise(default_value)
'''

df3.withColumn(
    "price_category",

    # price >= 100 → High
    when(col('price') >= 100,"High")
    # 50 <= price < 100 → Medium
    .when((col('price') < 100) & (col('price') >= 50),"Medium")

    # All remaining values → Low
    .otherwise("Low")
).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+--------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|price_category|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+--------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|    2017-09-19 09:45:35| 58.9|        13.29|2017|    9|        Medium|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|    2017-05-03 11:05:13|239.9|        19.93|2017|    5|          High|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|    2018-01-18 14:48:30|199.0|        17.87|2018|    1|          High|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|    2018-08-15 10:10:18|12.9

In [ ]:
# Import Window to define the partitioning and ordering rules
# used by window functions.
from pyspark.sql.window import Window


# Define a window specification:
# - Partition rows by year
# - Within each year, order rows by price in ascending order
windowSpec1 = Window.partitionBy('year').orderBy(
    col('price').asc()
)


# Apply dense_rank() using the window specification.
# Rows are ranked separately within each year based on price.
df3.withColumn(
    "dense_rank",
    dense_rank().over(windowSpec1)
).show(5)


# Define another window specification:
# - Partition rows by year
# - Within each year, order rows by shipping date/time
windowSpec2 = Window.partitionBy('year').orderBy(
    col('shipping_limit_datetime').asc()
)


# Calculate a running sum of price within each year,
# following the chronological order defined by windowSpec2.
# select() is used to display only the relevant columns.
df3.withColumn(
    'running_sum',
    sum('price').over(windowSpec2)
).select(
    'year',
    'price',
    'shipping_limit_datetime',
    'running_sum'
).show(5)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+----------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|dense_rank|
+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+----------+
|3ee6513ae7ea23bdf...|            1|8a3254bee785a526d...|96804ea39d96eb908...|    2018-05-04 03:55:26| 0.85|        18.23|2018|    5|         1|
|6e864b3f0ec710311...|            1|8a3254bee785a526d...|96804ea39d96eb908...|    2018-05-02 20:30:34| 0.85|        18.23|2018|    5|         1|
|c5bdd8ef3c0ec4202...|            2|8a3254bee785a526d...|96804ea39d96eb908...|    2018-05-07 02:55:22| 0.85|         22.3|2018|    5|         1|
|f1d5c2e6867fa93ce...|            1|46fce52cef5caa7cc...|2d2322d8421188677...|    2018-08-28 21:30:15|  2.2|         7.39|2018|   

+----+-----+-----------------------+------------------+
|year|price|shipping_limit_datetime|       running_sum|
+----+-----+-----------------------+------------------+
|2016| 59.5|    2016-09-19 00:15:34|              59.5|
|2016|44.99|    2016-09-19 23:11:33|194.47000000000003|
|2016|44.99|    2016-09-19 23:11:33|194.47000000000003|
|2016|44.99|    2016-09-19 23:11:33|194.47000000000003|
|2016|29.99|    2016-10-08 10:34:01|224.46000000000004|
+----+-----+-----------------------+------------------+
only showing top 5 rows



In [ ]:
# Define the path of the sellers CSV dataset.
hdfs_path = '/tmp/spark_datasets/sellers_dataset.csv'


# Read the CSV file into a Spark DataFrame.
# header=True → use the first row as column names.
# inferSchema=True → automatically infer the data types.
sdf = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load(hdfs_path)


# Display the inferred schema of the DataFrame.
sdf.printSchema()


# Display the first 5 rows of the DataFrame.
sdf.show(5)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
+--------------------+----------------------+-----------------+------------+
only showing top 5 rows



In [ ]:
# Join transformation


result1 = df3.join(broadcast(sdf), df3.seller_id == sdf.seller_id  , 'inner').drop(sdf.seller_id)

# Join transformation

# Syntax of Join: df1.join(df2, join_condition, join_type)

# Broadcast the sellers DataFrame because it is small enough
# to be distributed to the executors for the join.
# This can avoid shuffling the larger DataFrame during the join.
# We are using broadcast join here because the sellers dataset is small and can fit in memory. 
# This will improve the performance of the join operation.
# Also at the end we are dropping the seller_id column from the sellers dataset because it is already present in the 
# order_items dataset and we don't want to have duplicate columns in the final result.
#
# Join df3 and sdf using seller_id with an INNER JOIN.
# Both DataFrames contain seller_id, so drop the seller-side
# seller_id after the join to avoid keeping duplicate join columns.

result1 = df3.join(
    broadcast(sdf),
    df3.seller_id == sdf.seller_id,
    'inner'
).drop(sdf.seller_id)


# Multiple joins can also be chained together.
# Here, the result of the sellers join could be followed
# by a LEFT JOIN with the payment DataFrame.
# df3.join(
#     broadcast(sdf),
#     df3.seller_id == sdf.seller_id,
#     'inner'
# ).join(paydf, cond, 'left')


# Alternatively, perform the second join separately.
# result2 = result1.join(paydf, cond, 'left')


# Display the schema after the join.
result1.printSchema()

# Display the first 5 rows of the joined DataFrame.
result1.show(5)



root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_datetime: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+----------------------+-------------+------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|seller_zip_code_prefix|  seller_city|seller_state|
+--------------------+-------------+--------------------+--------------------+----------------------

In [ ]:
# Perform join using aliases for the DataFrames.
# Aliases make it easier to distinguish between columns
# with the same name in different DataFrames.

result2 = df3.alias('oid').join(
    sdf.alias('sid'),

    # Join using seller_id from each DataFrame.
    # oid.seller_id → seller_id from df3
    # sid.seller_id → seller_id from sdf
    col('oid.seller_id') == col('sid.seller_id'),

    # Keep only rows where a matching seller_id exists
    # in both DataFrames.
    'inner'

# Drop the seller_id column coming from sdf.
# The seller_id from df3 is retained to avoid duplicate columns.
).drop(
    col('sid.seller_id')
)


# Display the schema of the joined DataFrame.
result2.printSchema()

# Display the first 5 rows.
result2.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_datetime: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+----------------------+-------------+------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|seller_zip_code_prefix|  seller_city|seller_state|
+--------------------+-------------+--------------------+--------------------+----------------------

In [ ]:
# work with spark SQL

# We need to make sure that the dataframes are registered as temporary views before we can run SQL queries on them. 
# This is done using the createOrReplaceTempView method. Once the dataframes are registered as temporary views, 
# we can run SQL queries on them using the spark.sql method.


# Register the DataFrames as temporary SQL views.
# This allows us to reference the DataFrames by name
# inside Spark SQL queries.
df3.createOrReplaceTempView("ORDER_ITEM")
sdf.createOrReplaceTempView("SELLERS")

# This will also generate a physical plan for the query and optimize it before executing it. 
# The result of the query will be a new dataframe that contains the joined data from both tables.
# Execute a SQL query using the temporary views.
# ORDER_ITEM is aliased as "oid".
# SELLERS is aliased as "sid".
# Perform an INNER JOIN using seller_id from both views.
#
# spark.sql() returns the query result as a Spark DataFrame.
joinedDF = spark.sql("""
    SELECT *
    FROM ORDER_ITEM oid
    INNER JOIN SELLERS sid
        ON oid.seller_id = sid.seller_id
""")


# Display the schema of the resulting DataFrame.
joinedDF.printSchema()

# Display the first 5 rows.
joinedDF.show(5)


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_datetime: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------+-------------+--------------------+--------------------+-----------------------+-----+-------------+----+-----+--------------------+----------------------+-------------+------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_datetime|price|freight_value|year|month|           seller_id|seller_zip_code_prefix|  seller_city|seller_state|
+----------------

In [ ]:
# Write csv data in HDFS without any partition key

# Writing the data in csv format to the specified HDFS path. 
# The mode is set to 'overwrite' which means if the file already exists, it will be overwritten. 
# The header option is set to 'true' to include the column names in the output file, and the delimiter is set to ',' for CSV format.
result1.write \
    .format('csv') \
    .mode('overwrite') \
    .option('header', 'true') \
    .option('delimiter', ',') \
    .save('/tmp/spark_output/result1')

print("Write Successfull")

# Note: Important: this does NOT create one CSV file
# One Spark partition → generally one output part file.

Write Successfull


In [ ]:
# Write data in parquet format in HDFS without any partition key

# Writing the data in parquet format to the specified HDFS path. 
# The mode is set to 'overwrite' which means if file already exists, it will be overwritten. 
# Parquet is a columnar storage file format that is optimized for use with Spark and other big data processing frameworks.
# Also notice we don't have specify any kind of library to write parquet format, because spark has built-in support for parquet format.
result1.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('/tmp/spark_output/result_pq')

print("Write Successfull")

'''
Note:
-----
Parquet stores schema information with the data.

CSV essentially stores text values, so when reading it later, Spark may need:

.option("inferSchema", "true")

or an explicitly supplied schema.

Parquet already carries schema metadata.
'''

Write Successfull


In [ ]:
# Write data in HDFS with partition key

# Writing the data in csv format to the specified HDFS path with partitioning by the 'year' column. The mode is set to 'overwrite' 
# which means if the file already exists, it will be overwritten. The header option is set to 'true' to include the column names 
# in the output file, and the delimiter is set to ',' for CSV format. The partitionBy option is used to specify the column(s) 
# by which the data should be partitioned.
result1.write.\
    mode('overwrite').\
    partitionBy('year').\
    format('csv').\
    option('header', 'true').\
    option('delimiter', ',').\
    save('/tmp/spark_output/result2')
print("Write Successfull")

'''
result2/
│
├── year=2023/
│   ├── part-00000-....csv
│   └── ...
│
├── year=2024/
│   ├── part-00000-....csv
│   └── ...
│
└── year=2025/
    ├── part-00000-....csv
    └── ...

Spark can potentially avoid reading the directories for other years.

That's called partition pruning.
'''


Write Successfull


In [ ]:
# Write data in HDFS into single file

# Using the coalesce function to reduce the number of partitions to 1 before writing the data to HDFS. 
# This will result in a single output file being created in the specified HDFS path. The mode is set to 'overwrite' which means 
# if the file already exists, it will be overwritten. The header option is set to 'true' to include the column names in the output 
# file, and the delimiter is set to ',' for CSV format.
result1.coalesce(1) \
    .write \
    .mode('overwrite') \
    .format('csv') \
    .option('header', 'true') \
    .option('delimiter', ',') \
    .save('/tmp/spark_output/result3/')

print("Write Successfull")

Write Successfull


In [ ]:
# ============================================================
# 1. Read nested JSON data
# ============================================================

file_path = '/tmp/spark_datasets/orders_json_data.json'

# Read the JSON file into a Spark DataFrame.
# Spark automatically infers the JSON schema.
json_df = spark.read.json(file_path)

print("Original Schema:")
json_df.printSchema()

json_df.show(5)


# ============================================================
# 2. Explode the "purchases" array
# ============================================================

# Select the required user-level columns.
# explode() converts each element of the purchases array
# into a separate row.
#
# Each purchase is stored as a struct, so the resulting
# "purchase" column contains fields such as order_id,
# order_date and items.

purchases_df = json_df.select(
    "user_id",
    "name",
    "address",
    explode(col("purchases")).alias("purchase")
)

print("Exploded Schema:")
purchases_df.printSchema()

purchases_df.show(5)


# ============================================================
# 3. Explode the nested "items" array
# ============================================================

# Extract fields from the purchase struct.
# purchase.order_id → order_id
# purchase.order_date → order_date
#
# purchase.items is an array, so explode() creates
# one row for each item within the purchase.

items_df = purchases_df.select(
    "user_id",
    "name",
    "address",
    col("purchase.order_id").alias("order_id"),
    col("purchase.order_date").alias("order_date"),
    explode(col("purchase.items")).alias("item")
)

print("Fully Exploded Data:")
items_df.show(truncate=False)

Original Schema :
root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |-- name: string (nullable = true)
 |-- purchases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- item_name: string (nullable = true)
 |    |    |    |    |-- price: double (nullable = true)
 |    |    |-- order_date: string (nullable = true)
 |    |    |-- order_id: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- user_id: string (nullable = true)

+--------------------+-----------+--------------------+-----------------+-------+
|             address|       name|           purchases|registration_date|user_id|
+--------------------+-----------+--------------------+-----------------+-------+
|{Chicago, 311 Exa...|User Name